Importing libraries

In [98]:
import random

import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn

import pathlib as pl
import json

Handling Directories

In [99]:
root_dir = pl.Path("data")
dataset_file = root_dir / "iraqi_support_dataset_3000.json"

dataset_file.is_file()  # checking if file exists

True

In [100]:
with open(dataset_file) as file:
    test_data = json.load(file)

# printing first sample
test_data[random.randint(0, len(test_data))]

{'id': 345,
 'category': 'payment_pending',
 'priority': 'medium',
 'messages': [{'msg_id': 473, 'seq': 0, 'text': 'عملية الدفع مو حاسمة بعدها'}],
 'expected': {'amount': None,
  'transaction_id': None,
  'recipient': None,
  'account_id': None,
  'date': None}}

 Defining Acoustic Device

In [101]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.mps.is_available() else "cpu")

device

'mps'

### Modeling Pipeline

Categories

In [102]:
categories = [
    "failed_transfer",
    "wrong_recipient",
    "payment_pending",
    "card_issue",
    "login_problem",
    "agent_dispute",
    "wallet_topup",
    "refund_request",
    "account_issue",
    "other",
]

In [103]:
# Importing Transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [104]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
)

Loading weights: 100%|██████████| 434/434 [00:09<00:00, 44.28it/s]


In [105]:
#model to device
model = model.to(device)

Prompt builder

- as system
- as user

In [106]:
print(type(tokenizer))
print(type(model))

<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [107]:
def sys_prompt_build(categories: list) -> str:
    categories_text = "\n".join(f"-{c}" for c in categories)

    return f"""You are an Iraqi Arabic customer support ticket analyzer.
Analyze the full conversation.

Your tasks:
1. Classify the ticket into exactly one of the allowed categories.
2. Extract only information explicitly mentioned in the conversation.
3. Do not guess missing values.
4. Return valid JSON only.

Allowed categories:
{categories_text}

Extract these fields:
- amount
- transaction_id
- recipient
- account_id
- date

Return JSON using exactly this structure:

{{
  "category": "category_name",
  "entities": {{
    "amount": null,
    "transaction_id": null,
    "recipient": null,
    "account_id": null,
    "date": null
  }}
}}
    """


def users_prompt_build(messages: list) -> str:
    return "\n".join(f"Customer: {m["text"]}" for m in messages)

In [108]:
def tokenize(conv: list[dict]):
    text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt").to(device)

In [109]:
def build_conversation(complain: list, categories: list) -> list:
    return [
        {
            "role": "system",
            "content": sys_prompt_build(categories=categories)
        },
        {
            "role": "user",
            "content": users_prompt_build(complain)
        }
    ]

In [110]:
complain = test_data[0]["messages"]
conversation = build_conversation(complain=complain, categories=categories)

In [111]:
inputs = tokenize(conv=conversation)
inputs  # printing tokenized conv

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    458,  30234,  34117,   6002,
           1824,  11727,  54660,    624,   2082,  55856,    279,   2480,  10435,
            382,   7771,   9079,    510,     16,     13,   3228,   1437,    279,
          11727,   1119,   6896,    825,    315,    279,   5420,  11059,    624,
             17,     13,  22826,   1172,   1995,  20975,   9733,    304,    279,
          10435,    624,     18,     13,   3155,    537,   7942,   7402,   2750,
            624,     19,     13,   3411,   2697,   4718,   1172,    382,  35382,
          11059,    510,   2220,   5687,  35403,    198,   2630,  14347,   7080,
          14117,    198,  72443,  37861,    198,  20466,  53340,    198,  35048,
          60139,    198,  41935,   9932,   8492,    198,   2630,   7464,  10426,
            454,    198,  43007,   1241,   7893,    198,  49982,  53340,    198,
          72472,    271,  28959,   1493,   5043,    510,     12,   3311,    198,
             1

Generate LLM Output

In [112]:
def generate_llm_output(inputs):
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [142]:
sample_idx = random.randint(0, len(test_data) - 1)
sample = test_data[sample_idx]  # getting messages of complain
print(f"testing sample idx {sample_idx}, msg_id:{sample["id"]}")

for m in sample["messages"]:
    print(f"Message Content: {m["text"]}")

conversation = build_conversation(sample["messages"], categories)
inputs = tokenize(conv=conversation)

response = generate_llm_output(inputs)

testing sample idx 1897, msg_id:1898
Message Content: ظهر البطاقة ممسوح وشريطها ما يخلي الجهاز يقراها. حسابي اللي أستخدمه رقمه 88805547. أرجو ما ينفهم البلاغ على أنه طلب حساب جديد. الموضوع اللي أبلّغ عنه صار بتاريخ 20-09-2026


In [143]:
llm_extracted_data = json.loads(response)
llm_extracted_data

{'category': 'card_issue',
 'entities': {'amount': None,
  'transaction_id': None,
  'recipient': None,
  'account_id': '88805547',
  'date': '20-09-2026'}}